In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# SABRE를 활용한 트랜스파일 최적화

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*사용 시간 예상: Heron r2 프로세서에서 1분 미만 (참고: 이는 예상치이며 실제 런타임은 다를 수 있습니다.)*
## 배경
트랜스파일은 양자 회로를 특정 양자 하드웨어와 호환되는 형태로 변환하는 Qiskit의 중요한 단계입니다. 이 과정에는 두 가지 핵심 단계가 포함됩니다: **qubit 레이아웃** (논리적 qubit을 장치의 물리적 qubit에 매핑)과 **gate 라우팅** (SWAP gate를 필요에 따라 삽입하여 다중 qubit gate가 장치 연결성을 따르도록 보장).

SABRE (*SWAP 기반 양방향 휴리스틱 탐색 알고리즘*)는 레이아웃과 라우팅 모두에 강력한 최적화 도구입니다. **대규모 회로** (100개 이상의 qubit)와 가능한 qubit 매핑의 지수적 증가로 인해 효율적인 해결책이 필요한 **IBM&reg; Heron** 같은 복잡한 커플링 맵을 갖는 장치에 특히 효과적입니다.

### SABRE를 사용하는 이유
SABRE는 SWAP gate 수를 최소화하고 회로 깊이를 줄여 실제 하드웨어에서의 회로 성능을 향상시킵니다. 휴리스틱 기반 접근 방식은 고급 하드웨어와 크고 복잡한 회로에 이상적입니다. [LightSABRE](https://arxiv.org/abs/2409.08368) 알고리즘에서 도입된 최근 개선 사항은 더 빠른 런타임과 더 적은 SWAP gate를 제공하며 SABRE의 성능을 더욱 최적화합니다. 이러한 향상으로 대규모 회로에서 더욱 효과적입니다.

### 학습 내용
이 튜토리얼은 두 부분으로 나뉩니다:
1. 대규모 회로의 고급 최적화를 위해 **Qiskit 패턴**과 함께 SABRE 사용법 익히기.
2. 확장 가능하고 효율적인 트랜스파일을 위해 **qiskit_serverless**를 활용하여 SABRE의 잠재력 극대화.

다음을 배우게 됩니다:
- 100개 이상의 qubit를 가진 회로에 대한 SABRE 최적화를 통해 `optimization_level=3` 같은 기본 트랜스파일 설정 초과 달성.
- 런타임을 개선하고 gate 수를 줄이는 **LightSABRE 개선 사항** 탐색.
- **회로 품질**과 **트랜스파일 런타임**의 균형을 위해 SABRE 핵심 파라미터 (`swap_trials`, `layout_trials`, `max_iterations`, `heuristic`) 커스터마이징.
## 요구 사항
이 튜토리얼을 시작하기 전에 다음 항목이 설치되어 있는지 확인하세요:
- Qiskit SDK v1.0 이상, [시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함
- Qiskit Runtime v0.28 이상 (`pip install qiskit-ibm-runtime`)
- Serverless (`pip install qiskit-ibm-catalog qiskit_serverless`)
## 설정

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.transpiler.passes import (
    SabreLayout,
    SabreSwap,
    BarrierBeforeFinalMeasurements,
    StarPreRouting,
)
from qiskit.transpiler.passes.layout.vf2_layout import VF2LayoutStopReason
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.passmanager.flow_controllers import ConditionalController
import matplotlib.pyplot as plt
import numpy as np
import time

seed = 42

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token="<YOUR_API_TOKEN>",  # Replace with your actual API token
    instance="<YOUR_INSTANCE_NAME>",  # Replace with your instance name if needed
)
backend = service.least_busy(operational=True, simulator=False)


print(f"Using backend: {backend.name}")

Using backend: ibm_kingston


## Part I. Qiskit 패턴과 함께 SABRE 사용하기

SABRE는 Qiskit에서 qubit 레이아웃과 gate 라우팅 단계를 모두 처리하여 양자 회로를 최적화하는 데 사용할 수 있습니다. 이 섹션에서는 최적화 단계 2에 주로 초점을 맞추어 Qiskit 패턴과 함께 SABRE를 사용하는 **최소 예제**를 안내합니다.

SABRE를 실행하려면 다음이 필요합니다:
- 양자 회로의 **DAG** (방향성 비순환 그래프) 표현.
- qubit이 물리적으로 어떻게 연결되어 있는지 지정하는 Backend의 **커플링 맵**.
- 레이아웃과 라우팅을 최적화하기 위해 알고리즘을 적용하는 **SABRE 패스**.

이 부분에서는 **SabreLayout** 패스에 집중합니다. 이 패스는 SWAP gate 수를 최소화하면서 가장 효율적인 초기 레이아웃을 찾기 위해 레이아웃과 라우팅 시도를 모두 수행합니다. 중요하게도, `SabreLayout`은 단독으로도 SWAP gate를 가장 적게 추가하는 해결책을 저장함으로써 내부적으로 레이아웃과 라우팅을 모두 최적화합니다. 단, **SabreLayout**만 사용할 때는 SABRE의 휴리스틱을 변경할 수 없지만 `layout_trials` 수는 커스터마이징할 수 있습니다.

### Step 1: 고전적 입력을 양자 문제로 매핑

**GHZ (Greenberger-Horne-Zeilinger)** 회로는 모든 qubit이 `|0...0⟩` 또는 `|1...1⟩` 상태에 있는 얽힘 상태를 준비하는 양자 회로입니다. $n$ qubit에 대한 GHZ 상태는 수학적으로 다음과 같이 표현됩니다:
$$ |\text{GHZ}\rangle = \frac{1}{\sqrt{2}} \left( |0\rangle^{\otimes n} + |1\rangle^{\otimes n} \right) $$

다음을 적용하여 구성됩니다:
1. 첫 번째 qubit에 Hadamard gate를 적용하여 중첩 상태 생성.
2. 첫 번째 qubit과 나머지 qubit을 얽히게 하는 일련의 CNOT gate 적용.

이 예제에서는 선형 토폴로지 대신 의도적으로 **별형 토폴로지 GHZ 회로**를 구성합니다. 별형 토폴로지에서는 첫 번째 qubit이 "허브" 역할을 하며 다른 모든 qubit은 CNOT gate를 사용하여 직접 연결됩니다. 이 선택은 의도적인 것입니다: **선형 토폴로지 GHZ 상태**는 이론적으로 SWAP gate 없이 선형 커플링 맵에서 $O(N)$ 깊이로 구현될 수 있지만, SABRE는 100-qubit GHZ 회로를 Backend의 heavy-hex 커플링 맵의 하위 그래프에 매핑함으로써 최적 해결책을 쉽게 찾을 수 있습니다.

**별형 토폴로지 GHZ 회로**는 훨씬 더 어려운 문제를 제시합니다. SWAP gate 없이 여전히 이론적으로 $O(N)$ 깊이로 실행될 수 있지만, 이 해결책을 찾으려면 회로의 비선형 연결성으로 인해 훨씬 어려운 최적 초기 레이아웃을 식별해야 합니다. 이 토폴로지는 보다 복잡한 조건에서 레이아웃과 라우팅 성능에 대한 구성 파라미터의 영향을 보여주기 때문에 SABRE를 평가하기 위한 더 나은 테스트 케이스로 사용됩니다.

![ghz_star_topology.png](../docs/images/tutorials/transpilation-optimizations-with-sabre/ghz_star_topology.avif)

주목할 사항:
- **HighLevelSynthesis** 도구는 위 이미지에 표시된 것처럼 SWAP gate를 도입하지 않고 별형 토폴로지 GHZ 회로에 대한 최적 $O(N)$ 깊이 해결책을 생성할 수 있습니다.
- 대안적으로, **StarPrerouting** 패스는 SABRE의 라우팅 결정을 안내함으로써 깊이를 더 줄일 수 있지만 여전히 일부 SWAP gate를 도입할 수 있습니다. 그러나 StarPrerouting은 런타임을 증가시키고 초기 트랜스파일 프로세스에 통합이 필요합니다.

이 튜토리얼의 목적상, 런타임과 회로 깊이에 대한 SABRE 구성의 직접적인 영향을 고립시켜 강조하기 위해 HighLevelSynthesis와 StarPrerouting을 모두 제외합니다. 각 qubit 쌍에 대해 기댓값 $\langle Z_0 Z_i \rangle$를 측정함으로써 다음을 분석합니다:
- SABRE가 SWAP gate와 회로 깊이를 얼마나 잘 줄이는지.
- 이러한 최적화가 실행된 회로의 충실도에 미치는 영향 ($\langle Z_0 Z_i \rangle = 1$에서의 편차는 얽힘 손실을 나타냄).!

In [2]:
num_qubits_sim = 15

# Create star-topology GHZ circuit
qc_sim = QuantumCircuit(num_qubits_sim)
qc_sim.h(0)
for i in range(1, num_qubits_sim):
    qc_sim.cx(0, i)
qc_sim.measure_all()

# ZZ operators: Z on qubit 0 and qubit i, identity elsewhere
operator_strings_sim = [
    "Z" + "I" * i + "Z" + "I" * (num_qubits_sim - 2 - i)
    for i in range(num_qubits_sim - 1)
]
operators_sim = [SparsePauliOp(op) for op in operator_strings_sim]

다음으로, 시스템의 동작을 평가하기 위한 관심 연산자를 매핑합니다. 특히, `ZZ` 연산자를 사용하여 qubit이 서로 멀어질수록 얽힘이 어떻게 저하되는지 조사합니다. 이 분석은 매우 중요한데, 멀리 있는 qubit에 대한 기댓값 $\langle Z_0 Z_i \rangle$의 부정확성이 회로 실행에서 노이즈와 오류의 영향을 드러낼 수 있기 때문입니다. 이러한 편차를 연구함으로써 다양한 SABRE 구성에서 회로가 얽힘을 얼마나 잘 보존하는지, 그리고 SABRE가 하드웨어 제약의 영향을 얼마나 효과적으로 최소화하는지에 대한 통찰을 얻을 수 있습니다.

In [3]:
# Build the default pass manager (no modifications yet)
pm_1 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)

# Visualize the layout stage to see where SabreLayout sits
pm_1.layout.draw()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/b40fe1e0-41cd-4e8b-acb9-801872d35f1f-0.avif" alt="Output of the previous code cell" />

In the diagram above, the `SabreLayout` pass we want to customize lives inside the `ConditionalController` at position **[2]** of the layout stage. That controller does two things:

- It gates `SabreLayout` so it only runs when `VF2Layout` at [1] failed to find a perfect mapping (otherwise the perfect VF2 layout is kept).
- It precedes `SabreLayout` with a `BarrierBeforeFinalMeasurements` pass that protects measurements from being reordered during SabreLayout's internal routing.

If we just `replace(index=2, passes=sl_2)`, both behaviors are dropped. To keep them, we re-wrap our custom `SabreLayout` in the same `ConditionalController` (with the same condition and the protective barrier) before swapping it in.

**Step 2b: Build custom `SabreLayout` passes and replace the default.**

In [4]:
cmap = backend.coupling_map

# Custom SabreLayout passes with more aggressive search
sl_2 = SabreLayout(
    coupling_map=cmap,
    seed=seed,
    max_iterations=4,
    layout_trials=200,
    swap_trials=200,
)
sl_3 = SabreLayout(
    coupling_map=cmap,
    seed=seed,
    max_iterations=8,
    layout_trials=200,
    swap_trials=200,
)


# Same condition the preset uses: only run SabreLayout when VF2Layout did not
# find a perfect mapping. This preserves any perfect layout VF2 produced at [1].
def _vf2_match_not_found(property_set):
    if property_set["layout"] is None:
        return True
    return (
        property_set["VF2Layout_stop_reason"] is not None
        and property_set["VF2Layout_stop_reason"]
        is not VF2LayoutStopReason.SOLUTION_FOUND
    )


def wrap_sabre(sabre_pass):
    """Re-wrap a SabreLayout in the original ConditionalController + barrier."""
    return ConditionalController(
        [
            BarrierBeforeFinalMeasurements(
                "qiskit.transpiler.internal.routing.protection.barrier"
            ),
            sabre_pass,
        ],
        condition=_vf2_match_not_found,
    )


# Build two fresh pass managers and swap in the wrapped custom SabreLayout at index 2
pm_2 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_3 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_2.layout.replace(index=2, passes=wrap_sabre(sl_2))
pm_3.layout.replace(index=2, passes=wrap_sabre(sl_3))

# Build pm_star: default preset with StarPreRouting added to the init stage
pm_star = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_star.init += StarPreRouting()

# Visualize pm_3 after replacement (pm_2 has the same structure, only max_iterations differs)
pm_3.layout.draw()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/79075a21-8f36-4fd9-9d0d-bd0e97395b60-0.avif" alt="Output of the previous code cell" />

### Step 2: 양자 하드웨어 실행을 위한 문제 최적화
이 단계에서는 127개의 qubit를 가진 특정 양자 하드웨어 장치에서의 실행을 위해 회로 레이아웃을 최적화하는 데 집중합니다. 이것이 튜토리얼의 주요 초점으로, **SABRE 최적화와 트랜스파일**을 수행하여 최상의 회로 성능을 달성합니다. `SabreLayout` 패스를 사용하여 라우팅 중 SWAP gate의 필요성을 최소화하는 초기 qubit 매핑을 결정합니다. `coupling_map`을 대상 Backend에 전달함으로써 `SabreLayout`은 장치의 연결성 제약에 레이아웃을 적응시킵니다.

트랜스파일 프로세스에는 `optimization_level=3`으로 `generate_preset_pass_manager`를 사용하고 `SabreLayout` 패스를 다양한 구성으로 커스터마이징합니다. 목표는 **가장 낮은 크기 및/또는 깊이**를 가진 트랜스파일된 회로를 생성하는 설정을 찾아 SABRE 최적화의 영향을 입증하는 것입니다.

#### 회로 크기와 깊이가 중요한 이유
- **낮은 크기 (gate 수):** 연산 수를 줄여 오류가 누적될 기회를 최소화합니다.
- **낮은 깊이:** 전체 실행 시간을 단축하여 디코히어런스를 방지하고 양자 상태 충실도를 유지하는 데 중요합니다.

이러한 지표를 최적화함으로써 노이즈가 있는 양자 하드웨어에서 회로의 신뢰성과 실행 정확도를 향상시킵니다.
Backend를 선택합니다.

In [5]:
results_sim = {}
for name, pm in [
    ("pm_1 (4,20,20)", pm_1),
    ("pm_2 (4,200,200)", pm_2),
    ("pm_3 (8,200,200)", pm_3),
    ("pm_star (default + StarPreRouting)", pm_star),
]:
    t0 = time.time()
    tqc = pm.run(qc_sim)
    elapsed = time.time() - t0
    depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
    size = tqc.size()
    ops_mapped = [op.apply_layout(tqc.layout) for op in operators_sim]
    results_sim[name] = {
        "tqc": tqc,
        "ops": ops_mapped,
        "depth": depth,
        "size": size,
        "time": elapsed,
    }
    print(f"{name}: 2Q Depth {depth}, Size {size}, Time {elapsed:.2f}s")

# Print improvement relative to default (pm_1)
baseline = results_sim["pm_1 (4,20,20)"]
print("\nImprovement vs. default (pm_1):")
for name in [
    "pm_2 (4,200,200)",
    "pm_3 (8,200,200)",
    "pm_star (default + StarPreRouting)",
]:
    r = results_sim[name]
    depth_pct = (baseline["depth"] - r["depth"]) / baseline["depth"] * 100
    size_pct = (baseline["size"] - r["size"]) / baseline["size"] * 100
    print(f"  {name}: 2Q depth {depth_pct:+.1f}%, size {size_pct:+.1f}%")

pm_1 (4,20,20): 2Q Depth 38, Size 183, Time 0.01s
pm_2 (4,200,200): 2Q Depth 36, Size 183, Time 0.15s
pm_3 (8,200,200): 2Q Depth 30, Size 158, Time 0.16s
pm_star (default + StarPreRouting): 2Q Depth 26, Size 160, Time 0.01s

Improvement vs. default (pm_1):
  pm_2 (4,200,200): 2Q depth +5.3%, size +0.0%
  pm_3 (8,200,200): 2Q depth +21.1%, size +13.7%
  pm_star (default + StarPreRouting): 2Q depth +31.6%, size +12.6%


All three modified pass managers produced circuits with lower 2Q depth than the default. The aggressive SABRE configurations (`pm_2` and `pm_3`) trade longer transpilation time for a wider search, while `pm_star` leverages the star structure of the circuit and produces an even shallower result without paying any extra transpilation cost. The exact gains will vary from run to run, but the general trend is consistent: more SABRE trials and iterations let the heuristic search a wider space, and structure-aware passes like `StarPreRouting` can sidestep that search entirely when the circuit shape matches.

Even at this small scale (15 qubits), the room for improvement is enough that all three approaches beat the default. With larger circuits (100+ qubits), the search space grows dramatically and the benefits of both increased trials and structure-aware passes become much more pronounced, as the large-scale section will show.

In [6]:
pm_names = list(results_sim.keys())
depths = [results_sim[n]["depth"] for n in pm_names]
sizes = [results_sim[n]["size"] for n in pm_names]
times = [results_sim[n]["time"] for n in pm_names]
colors = ["#404080", "#2a9d8f", "#a8d05e", "#e29bdd"]
x = np.arange(len(pm_names))

fig, axs = plt.subplots(1, 3, figsize=(14, 5))

# 2Q Depth
bars = axs[0].bar(x, depths, color=colors)
axs[0].set_ylabel("2Q Depth", fontsize=11)
axs[0].set_title("Two-Qubit Gate Depth", fontsize=13)
axs[0].set_ylim(0, max(depths) * 1.2)
for bar, val in zip(bars, depths):
    axs[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(depths) * 0.02,
        str(val),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
for i in range(1, len(depths)):
    pct = (depths[0] - depths[i]) / depths[0] * 100
    if pct != 0:
        axs[0].text(
            bars[i].get_x() + bars[i].get_width() / 2,
            bars[i].get_height() / 2,
            f"{pct:+.0f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )

# Size
bars = axs[1].bar(x, sizes, color=colors)
axs[1].set_ylabel("Gate Count", fontsize=11)
axs[1].set_title("Circuit Size", fontsize=13)
axs[1].set_ylim(0, max(sizes) * 1.2)
for bar, val in zip(bars, sizes):
    axs[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(sizes) * 0.02,
        str(val),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
for i in range(1, len(sizes)):
    pct = (sizes[0] - sizes[i]) / sizes[0] * 100
    if abs(pct) > 0.1:
        axs[1].text(
            bars[i].get_x() + bars[i].get_width() / 2,
            bars[i].get_height() / 2,
            f"{pct:+.0f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )

# Time
bars = axs[2].bar(x, times, color=colors)
axs[2].set_ylabel("Time (s)", fontsize=11)
axs[2].set_title("Transpilation Time", fontsize=13)
axs[2].set_ylim(0, max(times) * 1.3)
for bar, val in zip(bars, times):
    axs[2].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(times) * 0.03,
        f"{val:.2f}s",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

for ax in axs:
    ax.set_xticks(x)
    ax.set_xticklabels(pm_names, fontsize=8, rotation=15)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle(
    "Transpilation quality vs. configuration",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/bf75c45a-2c3e-4ef6-8336-0b3f69e6e8fb-0.avif" alt="Output of the previous code cell" />

다양한 구성이 회로 최적화에 미치는 영향을 평가하기 위해 `SabreLayout` 패스에 대한 고유한 설정을 가진 세 가지 패스 관리자를 만들 것입니다. 이러한 구성은 회로 품질과 트랜스파일 시간 사이의 절충점을 분석하는 데 도움이 됩니다.

#### 핵심 파라미터
- **`max_iterations`**: 레이아웃을 정제하고 라우팅 비용을 줄이기 위한 전방-후방 라우팅 반복 횟수.
- **`layout_trials`**: SWAP gate를 최소화하는 것을 선택하여 테스트된 임의 초기 레이아웃의 수.
- **`swap_trials`**: 각 레이아웃에 대한 라우팅 시도 횟수로, 더 나은 라우팅을 위해 gate 배치를 정제합니다.

`layout_trials`와 `swap_trials`를 늘리면 더 철저한 최적화를 수행할 수 있지만 트랜스파일 시간이 증가합니다.

#### 이 튜토리얼의 구성
1. **`pm_1`**: `optimization_level=3`의 기본 설정.
   - `max_iterations=4`
   - `layout_trials=20`
   - `swap_trials=20`

2. **`pm_2`**: 더 나은 탐색을 위해 시도 횟수 증가.
   - `max_iterations=4`
   - `layout_trials=200`
   - `swap_trials=200`

3. **`pm_3`**: 추가 정제를 위해 반복 횟수를 늘려 `pm_2`를 확장.
   - `max_iterations=8`
   - `layout_trials=200`
   - `swap_trials=200`

이러한 구성의 결과를 비교함으로써 회로 품질 (예: 크기와 깊이)과 계산 비용 사이에서 최상의 균형을 달성하는 것이 어느 것인지 결정하려 합니다.

In [15]:
# Create a noisy estimator from the real backend's noise model
noisy_estimator = AerEstimator.from_backend(backend)

num_runs = 10
# sim_all_runs[name] = list of arrays, one per run
sim_all_runs = {name: [] for name in results_sim}

for run in range(num_runs):
    for name, r in results_sim.items():
        job = noisy_estimator.run([(r["tqc"], r["ops"])])
        evs = list(job.result()[0].data.evs)
        sim_all_runs[name].append(evs)
    print(f"Run {run + 1}/{num_runs} done")

# Compute mean and std across runs for each config
sim_stats = {}
for name in results_sim:
    all_evs = np.array(sim_all_runs[name])  # shape (num_runs, num_operators)
    sim_stats[name] = {
        "mean": np.mean(all_evs, axis=0),
        "std": np.std(all_evs, axis=0),
        "overall_mean": np.mean(all_evs),
        "overall_std": np.std(
            np.mean(all_evs, axis=1)
        ),  # std of per-run averages
    }
    print(
        f"{name}: mean fidelity = {sim_stats[name]['overall_mean']:.4f} +/- {sim_stats[name]['overall_std']:.4f}"
    )

Run 1/10 done
Run 2/10 done
Run 3/10 done
Run 4/10 done
Run 5/10 done
Run 6/10 done
Run 7/10 done
Run 8/10 done
Run 9/10 done
Run 10/10 done
pm_1 (4,20,20): mean fidelity = 0.9510 +/- 0.0094
pm_2 (4,200,200): mean fidelity = 0.9513 +/- 0.0043
pm_3 (8,200,200): mean fidelity = 0.9540 +/- 0.0065
pm_star (default + StarPreRouting): mean fidelity = 0.9547 +/- 0.0072


이제 커스텀 패스 관리자에서 `SabreLayout` 패스를 구성할 수 있습니다. 이를 위해 `optimization_level=3`에서 기본 `generate_preset_pass_manager`의 경우 `SabreLayout` 패스가 인덱스 2에 있음을 알고 있습니다. `SabreLayout`이 `SetLayout` 및 `VF2Layout` 패스 다음에 발생하기 때문입니다. 이 패스에 접근하여 파라미터를 수정할 수 있습니다.

In [16]:
data_sim = list(range(1, len(operators_sim) + 1))
markers = ["o", "s", "^", "*"]
colors_line = ["#404080", "#2a9d8f", "#a8d05e", "#e29bdd"]

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.5, 1]}
)

# Left: correlations vs distance with error bars (mean +/- 1 std)
for (name, stats), marker, color in zip(
    sim_stats.items(), markers, colors_line
):
    ax1.errorbar(
        data_sim,
        stats["mean"],
        yerr=stats["std"],
        marker=marker,
        label=name,
        color=color,
        linewidth=2,
        capsize=3,
        capthick=1,
        elinewidth=1,
    )

ax1.set_xlabel("Distance between qubits $i$", fontsize=11)
ax1.set_ylabel(r"$\langle Z_0 Z_i \rangle$", fontsize=11)
ax1.set_title(
    "Entanglement correlations vs. qubit distance (avg. of 10 runs)",
    fontsize=12,
)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Right: mean correlation bar chart with error bars
names = list(sim_stats.keys())
means = [sim_stats[n]["overall_mean"] for n in names]
stds = [sim_stats[n]["overall_std"] for n in names]
x_bar = np.arange(len(names))
bars = ax2.bar(
    x_bar, means, yerr=stds, color=colors_line, capsize=5, ecolor="gray"
)
ax2.set_ylabel(r"Mean $\langle Z_0 Z_i \rangle$", fontsize=11)
ax2.set_title("Average fidelity", fontsize=13, pad=12)
y_range = max(means) - min(means) if max(means) != min(means) else 0.01
# Top of ylim accounts for the bar height + std error bar + headroom for the value label
y_top = max(m + s for m, s in zip(means, stds)) + y_range * 1.5
ax2.set_ylim(min(means) - y_range * 0.8, y_top)
for bar, val, std in zip(bars, means, stds):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + y_range * 0.15,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )
# Annotate % change vs pm_1
baseline_mean = means[0]
for i in range(1, len(means)):
    pct = (means[i] - baseline_mean) / baseline_mean * 100
    if abs(pct) > 0.01:
        mid_y = (means[i] + ax2.get_ylim()[0]) / 2
        ax2.text(
            bars[i].get_x() + bars[i].get_width() / 2,
            mid_y,
            f"{pct:+.1f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )
ax2.set_xticks(x_bar)
ax2.set_xticklabels(names, fontsize=8, rotation=15)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

fig.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/a6dac5ed-a963-458a-ada1-89c915f036e0-0.avif" alt="Output of the previous code cell" />

각 패스 관리자가 구성되면 각각에 대한 트랜스파일 프로세스를 실행합니다. 결과를 비교하기 위해 트랜스파일 시간, 회로 깊이 (2-qubit gate 깊이로 측정), 트랜스파일된 회로의 총 gate 수 등 핵심 지표를 추적합니다.

In [9]:
# -------------------------Step 1-------------------------

num_qubits = 100

# Create star-topology GHZ circuit
qc = QuantumCircuit(num_qubits)
qc.h(0)
for i in range(1, num_qubits):
    qc.cx(0, i)
qc.measure_all()

# ZZ operators
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (num_qubits - 2 - i)
    for i in range(num_qubits - 1)
]
operators = [SparsePauliOp(op) for op in operator_strings]

In [10]:
# -------------------------Step 2-------------------------

num_seeds = 10
seed_list = [seed + i for i in range(num_seeds)]
swap_trials = 200


# The default routing[1] is a ConditionalController([barrier, routing_pass],
# condition=_swap_condition); we re-wrap so the new routing pass keeps the
# protective barrier and is skipped when routing isn't needed (matches the preset).
def _swap_condition(property_set):
    return not property_set["routing_not_needed"]


def wrap_routing(routing_pass):
    return ConditionalController(
        [
            BarrierBeforeFinalMeasurements(
                "qiskit.transpiler.internal.routing.protection.barrier"
            ),
            routing_pass,
        ],
        condition=_swap_condition,
    )


heuristic_results = {}

# Three SABRE heuristics, swept over seeds
for heuristic in ["basic", "decay", "lookahead"]:
    trials = []
    for s in seed_list:
        sr = SabreSwap(
            coupling_map=cmap, heuristic=heuristic, trials=swap_trials, seed=s
        )
        sl = SabreLayout(coupling_map=cmap, routing_pass=sr, seed=s)
        pm = generate_preset_pass_manager(
            optimization_level=3, backend=backend, seed_transpiler=s
        )
        # Re-wrap each custom pass in its original ConditionalController + barrier
        # (wrap_sabre is defined in the small-scale Step 2 cell above).
        pm.layout.replace(index=2, passes=wrap_sabre(sl))
        pm.routing.replace(index=1, passes=wrap_routing(sr))

        t0 = time.time()
        tqc = pm.run(qc)
        elapsed = time.time() - t0
        depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
        size = tqc.size()
        trials.append(
            {
                "tqc": tqc,
                "depth": depth,
                "size": size,
                "time": elapsed,
                "seed": s,
            }
        )

    heuristic_results[heuristic] = trials

# Default preset + StarPreRouting in init, also swept over seeds for a fair comparison
star_trials = []
for s in seed_list:
    pm_star_hw = generate_preset_pass_manager(
        optimization_level=3, backend=backend, seed_transpiler=s
    )
    pm_star_hw.init += StarPreRouting()

    t0 = time.time()
    tqc = pm_star_hw.run(qc)
    elapsed = time.time() - t0
    depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
    size = tqc.size()
    star_trials.append(
        {
            "tqc": tqc,
            "depth": depth,
            "size": size,
            "time": elapsed,
            "seed": s,
        }
    )
heuristic_results["StarPreRouting"] = star_trials

# Print summary for each entry
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    trials = heuristic_results[label]
    depths = [t["depth"] for t in trials]
    sizes = [t["size"] for t in trials]
    best = min(trials, key=lambda t: t["depth"])
    print(f"{label}:")
    print(
        f"  2Q depth: min: {min(depths)}, mean: {np.mean(depths):.1f}, std: {np.std(depths):.1f}"
    )
    print(
        f"  size    : min: {min(sizes)}, mean: {np.mean(sizes):.1f}, std: {np.std(sizes):.1f}"
    )
    print(
        f"  best seed: {best['seed']} (2Q depth={best['depth']}, size={best['size']})"
    )

basic:
  2Q depth: min: 524, mean: 570.5, std: 39.9
  size    : min: 3819, mean: 4227.1, std: 360.6
  best seed: 51 (2Q depth=524, size=3852)
decay:
  2Q depth: min: 387, mean: 436.4, std: 41.7
  size    : min: 2687, mean: 3183.1, std: 459.3
  best seed: 45 (2Q depth=387, size=2786)
lookahead:
  2Q depth: min: 364, mean: 424.6, std: 36.5
  size    : min: 2335, mean: 3014.6, std: 388.1
  best seed: 51 (2Q depth=364, size=2485)
StarPreRouting:
  2Q depth: min: 196, mean: 196.0, std: 0.0
  size    : min: 1151, mean: 1151.0, std: 0.0
  best seed: 42 (2Q depth=196, size=1151)


In [11]:
hw_colors = {
    "basic": "#ff7f0e",
    "decay": "#d62728",
    "lookahead": "#1f77b4",
    "StarPreRouting": "#2a9d8f",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    trials = heuristic_results[label]
    depths = [t["depth"] for t in trials]
    sizes = [t["size"] for t in trials]
    seeds = [t["seed"] for t in trials]
    color = hw_colors[label]

    ax1.scatter(
        seeds,
        depths,
        label=label,
        color=color,
        alpha=0.8,
        edgecolor="k",
        s=60,
    )
    ax1.axhline(np.mean(depths), color=color, linestyle="--", alpha=0.5)

    ax2.scatter(
        seeds,
        sizes,
        label=label,
        color=color,
        alpha=0.8,
        edgecolor="k",
        s=60,
    )
    ax2.axhline(np.mean(sizes), color=color, linestyle="--", alpha=0.5)

ax1.set_xlabel("Seed", fontsize=11)
ax1.set_ylabel("2Q Depth", fontsize=11)
ax1.set_title("Two-Qubit Gate Depth per Seed", fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

ax2.set_xlabel("Seed", fontsize=11)
ax2.set_ylabel("Gate Count", fontsize=11)
ax2.set_title("Circuit Size per Seed", fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.suptitle(
    "Transpilation variability across seeds: SABRE heuristics vs. StarPreRouting",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

# Summary comparison
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    best = min(heuristic_results[label], key=lambda t: t["depth"])
    print(
        f"{label}: best 2Q depth={best['depth']}, size={best['size']} (seed={best['seed']})"
    )

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/eead9bd2-17e0-4f5b-80bc-eb9b30af052e-0.avif" alt="Output of the previous code cell" />

basic: best 2Q depth=524, size=3852 (seed=51)
decay: best 2Q depth=387, size=2786 (seed=45)
lookahead: best 2Q depth=364, size=2485 (seed=51)
StarPreRouting: best 2Q depth=196, size=1151 (seed=42)


In [12]:
# -------------------------Step 3: Execute on hardware-------------------------

best_circuits = {}
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    best_circuits[label] = min(
        heuristic_results[label], key=lambda t: t["depth"]
    )
    b = best_circuits[label]
    print(f"Best {label}: 2Q depth={b['depth']}, size={b['size']}")

options = EstimatorOptions()
options.resilience_level = 2
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"
estimator = Estimator(backend, options=options)

hw_jobs = {}
hw_ops = {}
for label, best in best_circuits.items():
    hw_ops[label] = [op.apply_layout(best["tqc"].layout) for op in operators]
    hw_jobs[label] = estimator.run([(best["tqc"], hw_ops[label])])
    print(f"{label} job: {hw_jobs[label].job_id()}")
estimator.options.environment.job_tags = ["TUT_TOWS"]

hw_results = {}
for label, job in hw_jobs.items():
    hw_results[label] = job.result()[0]
    print(f"{label} job done")

Best basic: 2Q depth=524, size=3852
Best decay: 2Q depth=387, size=2786
Best lookahead: 2Q depth=364, size=2485
Best StarPreRouting: 2Q depth=196, size=1151
basic job: d81q5tnoha1c73bknprg
decay job: d81q5tugbeec73aktopg
lookahead job: d81q5to0bvlc73d1epe0
StarPreRouting job: d81q5u7tjchs73bn82hg
basic job done
decay job done
lookahead job done
StarPreRouting job done


In [13]:
# -------------------------Step 4: Post-process-------------------------

data = list(range(1, len(operators) + 1))
hw_markers = {
    "basic": "D",
    "decay": "o",
    "lookahead": "s",
    "StarPreRouting": "*",
}
hw_labels = ["basic", "decay", "lookahead", "StarPreRouting"]

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.5, 1]}
)

# Left: correlations vs distance
for label in hw_labels:
    evs = list(hw_results[label].data.evs)
    b = best_circuits[label]
    ax1.plot(
        data,
        evs,
        marker=hw_markers[label],
        color=hw_colors[label],
        linewidth=2,
        label=f"{label} (2Q depth={b['depth']}, size={b['size']})",
        markersize=5 if label == "StarPreRouting" else 4,
    )

ax1.set_xlabel("Distance between qubits $i$", fontsize=11)
ax1.set_ylabel(r"$\langle Z_0 Z_i \rangle$", fontsize=11)
ax1.set_title(
    "Entanglement correlations vs. qubit distance (hardware)", fontsize=12
)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Right: mean fidelity bar chart
hw_means = [np.mean(list(hw_results[label].data.evs)) for label in hw_labels]
hw_bar_colors = [hw_colors[label] for label in hw_labels]
x_bar = np.arange(len(hw_labels))
bars = ax2.bar(x_bar, hw_means, color=hw_bar_colors)
ax2.set_ylabel(r"Mean $\langle Z_0 Z_i \rangle$", fontsize=11)
ax2.set_title("Average fidelity", fontsize=13)
y_range = (
    max(hw_means) - min(hw_means) if max(hw_means) != min(hw_means) else 0.01
)
ax2.set_ylim(min(hw_means) - y_range * 0.2, max(hw_means) + y_range * 0.15)
for bar, val in zip(bars, hw_means):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_range * 0.05,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
ax2.set_xticks(x_bar)
ax2.set_xticklabels(hw_labels, fontsize=9, rotation=15)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

fig.tight_layout()
plt.show()

print("\nMean fidelity:")
for label, m in zip(hw_labels, hw_means):
    print(f"  {label}: {m:.4f}")

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/0280b0b9-6320-43e5-8396-f82f9e718319-0.avif" alt="Output of the previous code cell" />


Mean fidelity:
  basic: 0.0344
  decay: 0.1298
  lookahead: 0.1857
  StarPreRouting: 0.3295


### Analysis

The scatter plots show significant variability across seeds for all three SABRE heuristics, which underscores the importance of running multiple layout trials rather than relying on a single transpilation. The `StarPreRouting` line is essentially flat across seeds because the rewrite from a star into a linear chain is deterministic given the structure; the downstream SABRE routing then has very little freedom on a linear chain, so the seed has almost no effect on the final depth or size.

From the transpilation results, both the `decay` and `lookahead` heuristics consistently outperform `basic` by a wide margin. The `basic` heuristic, while fast, uses a simple greedy strategy that often leads to substantially deeper circuits. For this star-topology GHZ circuit, `lookahead` tends to produce the lowest 2Q depth and gate count among the SABRE heuristics, since its forward-looking cost function is well suited to circuits with long-range connectivity patterns. `StarPreRouting`, however, dwarfs all three by a substantial margin: by rewriting the star into a linear chain before routing, it short-circuits the search problem entirely and delivers a circuit that the rest of the transpiler can map onto a linear path with minimal additional SWAPs.

That advantage carries straight over to hardware fidelity. Lower 2Q depth and gate count do not always translate one-for-one to higher fidelity (the specific physical qubits a layout uses and their calibration at run time also matter), but when the depth gap is as large as the one between SABRE and `StarPreRouting` here, the structure-aware approach wins decisively because the circuit accumulates far less decoherence and far fewer two-qubit error events. The fidelity bar chart shows `StarPreRouting` substantially ahead of even the best SABRE heuristic, while `basic` sits well below the rest because its much deeper circuits accumulate the most error.

**Key takeaways:**
- Among SABRE heuristics, `decay` and `lookahead` are substantially better than `basic` for non-trivial circuits. Prefer one of the two for production workloads.
- The best SABRE heuristic depends on your circuit and hardware. Testing multiple heuristics with multiple seeds is the most reliable strategy.
- If you want to explore even more layouts, increase `swap_trials` (and `layout_trials` when you are not pinning a custom routing pass) rather than fanning the work out to remote nodes. The SABRE passes already parallelize trials across local threads, and the per-trial work is small enough that distribution overhead typically dominates any speedup.
- When the circuit has a known special structure, applying a structure-aware pass like `StarPreRouting` before SABRE can deliver an order-of-magnitude improvement that no amount of SABRE tuning will match. This is not a replacement for SABRE: `StarPreRouting` only helps when the circuit actually contains star sub-circuits and the backend has a long enough linear path. It is worth checking the pass library for matches whenever you know your circuit's shape.

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [`SabreLayout` API reference](/docs/api/qiskit/qiskit.transpiler.passes.SabreLayout): full parameter documentation
- [SABRE paper](https://arxiv.org/abs/1809.02573): the original SABRE algorithm for layout and routing
- [LightSABRE paper](https://arxiv.org/abs/2409.08368): the algorithmic improvements that power Qiskit's current SABRE implementation
- [Write a custom transpiler pass](/docs/guides/custom-transpiler-pass): build your own transpilation logic
- [Transpiler plugins](/docs/guides/transpiler-plugins): extend Qiskit's transpilation pipeline with third-party passes
- [DAG representation](/docs/guides/DAG-representation): understand the directed acyclic graph used internally by the transpiler
</Admonition>

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_d9YWUSQIAvU9HXE)